# Crear la Base de Datos

In [1]:
import pandas as pd # Te sirve para manejar bases de datos
import os

In [24]:
data_dir = "../all_data/"
databases_dir = "../databases/"
archivos = os.listdir(data_dir)

In [3]:
# Tipos de dato
tipos_de_dato = {
    'cs_p13_1': int,
    'eda7c': int,
    'sex': int, 
    'e_con': int,
    'rama': int, 
    'ambito2': int, 
    'emp_ppal': int,
    'pos_ocu': int,
    'ing_x_hrs': float,
    'hrsocup': int 
}

In [4]:
columnas = ['cs_p13_1', 'eda7c', 'sex', 'e_con'] + \
           ['rama', 'ambito2', 'emp_ppal', 'pos_ocu'] + \
           ['ing_x_hrs', 'hrsocup']

database = pd.DataFrame(columns = columnas + ['ent', 'ano', 'trim'])
database = database.astype(tipos_de_dato)

In [5]:
# Lista de las entidades que queremos incluir
lista_de_entidades = {
    5: 'Coahuila', 
    9: 'Ciudad de México',
    7: 'Chiapas',
    19: 'Nuevo León',
    23: 'Quintana Roo'
}

In [6]:
# Leer todos los archivos
for archivo in archivos:
    # Leer archivo csv
    datos_tmp = pd.read_csv(data_dir + archivo,
                            encoding='latin-1')
    datos_tmp.columns = [col.lower() for col in datos_tmp.columns]
    datos_tmp = datos_tmp.rename({'cve_ent': 'ent'}, axis=1)
    datos_tmp = datos_tmp[columnas + ['ent']] # Solo columnas útiles
    

    # Conversión a datos numéricos
    for columna in columnas:
        datos_tmp[columna] = pd.to_numeric(datos_tmp[columna], errors='coerce')
    datos_tmp.dropna(inplace = True)
    datos_tmp = datos_tmp.astype(tipos_de_dato)

    # Agregar columnas de año y trimestre
    datos_tmp['ano'] = int(archivo.split('_')[-2])
    datos_tmp['trim'] = int(archivo.split('_')[-1][0])

    # Eliminar los elementos con ingreso 0
    datos_tmp = datos_tmp[datos_tmp['ing_x_hrs'] != 0]
    # Conservar solo los elementos de las entidades de interés
    # print(datos_tmp.columns)
    datos_tmp = datos_tmp[datos_tmp['ent'].isin(list(lista_de_entidades.keys()))]

    # Concatenar al dataframe grande
    database = pd.concat([database, datos_tmp], 
                         ignore_index = True)

C:\Users\juanj\AppData\Local\Temp\ipykernel_5032\440479194.py:4: DtypeWarning: Columns (2,5,8,22,23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_tmp = pd.read_csv(data_dir + archivo,
C:\Users\juanj\AppData\Local\Temp\ipykernel_5032\440479194.py:4: DtypeWarning: Columns (2,5,8,22,23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_tmp = pd.read_csv(data_dir + archivo,
C:\Users\juanj\AppData\Local\Temp\ipykernel_5032\440479194.py:4: DtypeWarning: Columns (2,5,8,22,23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_tmp = pd.read_csv(data_dir + archivo,
C:\Users\juanj\AppData\Local\Temp\ipykernel_5032\440479194.py:4: DtypeWarning: Columns (2,22,23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_tmp = pd.read_csv(data_dir + archivo,
C:\Users\juanj\AppData\Local\Temp\ipykernel_5032\440479194.p

In [7]:
database.head()

,cs_p13_1,eda7c,sex,e_con,rama,ambito2,emp_ppal,pos_ocu,ing_x_hrs,hrsocup,ent,ano,trim
0,3,3,1,1,4,7,2,1,52.09302,25,9,2020,3
1,3,2,1,6,3,4,2,1,65.11628,25,9,2020,3
2,3,2,1,6,3,3,2,1,73.64341,24,9,2020,3
3,4,5,1,5,4,2,1,3,66.66667,9,9,2020,3
4,6,4,2,6,2,3,1,1,37.50000,24,9,2020,3


In [8]:
database_backup = database.copy()

## Limpiar y Reformatear la Base de Datos

In [9]:
# Eliminar datos invalidos
database = database[(database['cs_p13_1'] != 0) & (database['cs_p13_1'] != 99)]

# Reformatear cs_p13_1
dict_esc = {
    1: 1,
    2: 1,
    3: 1,
    4: 2,
    5: 2,
    6: 2,
    7: 3,
    8: 4,
    9: 4
}

database['cs_p13_1'] = database['cs_p13_1'].replace(dict_esc).astype(int)

In [10]:
# Eliminar datos invalidos de eda7c
database = database[(database['eda7c'] != 0) & (database['eda7c'] != 7)]

In [11]:
# Eliminar datos invalidos
database = database[database['e_con'] != 9]

# Reformatear e_con
dict_e_con = {
    1: 1,
    2: 2,
    3: 2,
    4: 2,
    5: 1,
    6: 3
}

database['e_con'] = database['e_con'].replace(dict_e_con).astype(int)

In [12]:
# Eliminar datos invalidos y 'Otros'
database = database[(database['rama'] != 0) & (database['rama'] != 7) & (database['rama'] != 5)]

# Reformatear rama
dict_rama = {
    1: 2,
    2: 2,
    3: 3,
    4: 3,
    6: 1
}

database['rama'] = database['rama'].replace(dict_rama).astype(int)

In [13]:
# Eliminar datos invalidos
database = database[database['ambito2'] != 0]

# Reformatear ambito2
dict_ambito = {
    1: 1,
    2: 1,
    3: 2,
    4: 2,
    5: 2,
    6: 3,
    7: 4
}

database['ambito2'] = database['ambito2'].replace(dict_ambito).astype(int)

In [14]:
# Eliminar datos inválidos
database = database[database['emp_ppal'] != 0]

In [15]:
# Eliminar datos invalidos y 'trabajadores sin sueldo'
database = database[(database['pos_ocu'] != 0) & (database['pos_ocu'] != 4) & (database['pos_ocu'] != 5)]

In [ ]:
# Formatear columnas como ints
database = database.astype({'ent': int,
                            'ano': int,
                            'trim': int})

In [22]:
database.head()

,cs_p13_1,eda7c,sex,e_con,rama,ambito2,emp_ppal,pos_ocu,ing_x_hrs,hrsocup,ent,ano,trim
0,1,3,1,1,3,4,2,1,52.09302,25,9,2020,3
1,1,2,1,3,3,2,2,1,65.11628,25,9,2020,3
2,1,2,1,3,3,2,2,1,73.64341,24,9,2020,3
3,2,5,1,1,3,1,1,3,66.66667,9,9,2020,3
4,2,4,2,3,2,2,1,1,37.50000,24,9,2020,3


In [26]:
database.describe()

,cs_p13_1,eda7c,sex,e_con,rama,ambito2,emp_ppal,pos_ocu,ing_x_hrs,hrsocup,ent,ano,trim
count,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06,1.502149e+06
mean,1.741002e+00,3.320203e+00,1.400798e+00,1.665227e+00,2.728614e+00,2.346756e+00,1.544698e+00,1.407535e+00,4.356658e+01,4.404573e+01,1.203602e+01,2.015013e+03,2.502610e+00
std,8.422604e-01,1.315314e+00,4.900603e-01,8.845338e-01,4.446748e-01,1.543471e+00,4.979983e-01,7.783123e-01,6.648547e+01,1.667100e+01,7.111387e+00,6.218843e+00,1.124765e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,7.750000e-03,1.000000e+00,5.000000e+00,2.005000e+03,1.000000e+00
25%,1.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00,1.875000e+01,3.600000e+01,5.000000e+00,2.009000e+03,1.000000e+00
50%,2.000000e+00,3.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,3.000000e+01,4.700000e+01,9.000000e+00,2.015000e+03,3.000000e+00
75%,2.000000e+00,4.000000e+00,2.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,2.000000e+00,1.000000e+00,5.000000e+01,5.100000e+01,1.900000e+01,2.021000e+03,4.000000e+00
max,4.000000e+00,6.000000e+00,2.000000e+00,3.000000e+00,3.000000e+00,8.000000e+00,2.000000e+00,3.000000e+00,2.990033e+04,1.680000e+02,2.300000e+01,2.025000e+03,4.000000e+00


In [25]:
database.to_csv(databases_dir + 'full_database.csv', index=False)

## Dividir la base de datos

In [ ]:
database = database.drop('trim', axis=1)

,cs_p13_1,eda7c,sex,e_con,rama,ambito2,emp_ppal,pos_ocu,ing_x_hrs,hrsocup,ent,ano
0,1,3,1,1,3,4,2,1,52.09302,25,9,2020
1,1,2,1,3,3,2,2,1,65.11628,25,9,2020
2,1,2,1,3,3,2,2,1,73.64341,24,9,2020
3,2,5,1,1,3,1,1,3,66.66667,9,9,2020
4,2,4,2,3,2,2,1,1,37.50000,24,9,2020


In [43]:
table_ids = database[['ent', 'ano']].drop_duplicates().values

for table_id in table_ids:
    tmp_table  = database[(database['ent'] == table_id[0]) & (database['ano'] == table_id[1])]
    tmp_table.to_csv(databases_dir + f'enoe_{table_id[0]:02}_{table_id[1]:04}.csv', index=False)